In [0]:
#importamos librerias

from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import countDistinct
from pyspark.ml.feature import StringIndexer
import mlflow
spark.conf.set("spark.databricks.execution.timeout", 1800)

In [0]:
#cargamos el dataset de fraudes bancarios

df = spark.table('forecast.default.data')
df.describe().show()
df.printSchema()
df.select(countDistinct("type")).show()
df.select(countDistinct("nameOrig")).show()
df.show(10)


In [0]:
# Asegurar tipo numérico
df = df.withColumn("isFraud", col("isFraud").cast("double"))


#indexamos 'type'
indexer = StringIndexer(
    inputCol="type",
    outputCol="type_idx",
    handleInvalid="keep"
)

#features
numeric_features = [
    'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'
]

feature_cols = numeric_features + ["type_idx"]

# Vector de features
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df = indexer.fit(df).transform(df)

df_model = assembler.transform(df).select("features", "isFraud")

# Renombrar a label (Spark lo espera así)
df_model = df_model.withColumnRenamed("isFraud", "label")

# Split
train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42077651)

# Evaluadores
acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
auc_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Grid
param_grid = [
    {"numTrees": 50, "maxDepth": 3},
    {"numTrees": 100, "maxDepth": 5},
    {"numTrees": 150, "maxDepth": 7},
    {"numTrees": 200, "maxDepth": 9}
]

mlflow.set_experiment("/Users/martoschelp@gmail.com/RandomForest_Fraud")

for params in param_grid:
    with mlflow.start_run():
        rf = RandomForestClassifier(
            featuresCol="features",
            labelCol="label",
            numTrees=params["numTrees"],
            maxDepth=params["maxDepth"],
            seed=42
        )

        model = rf.fit(train_df)
        preds = model.transform(test_df)

        acc = acc_eval.evaluate(preds)
        auc = auc_eval.evaluate(preds)

        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)